In [1]:
from pathlib import Path
import requests
import wfdb

base = Path("data/mit-bih-arrhythmia-database-1.0.0")
record_id = "100"
record_path = str(base / record_id)

record = wfdb.rdrecord(record_path)
ann = wfdb.rdann(record_path, "atr")

for sample, symbol in zip(ann.sample, ann.symbol):
    if symbol in ["N", "A", "L", "R", "V"] and 180 < sample < len(record.p_signal) - 180:
        window = record.p_signal[sample - 180:sample + 180, 0].tolist()
        break

payload = {
    "signal": window
}

response = requests.post(
    "http://127.0.0.1:8001/predict",
    json=payload,
    timeout=60
)

print("Registro:", record_id)
print("Label real da anotação:", symbol)
print("R-peak:", sample)
print(response.json())

Registro: 100
Label real da anotação: N
R-peak: 370
{'model': '/Users/gubscruz/INSPER/7_PERIODO/ai-medicine/ecg-anomaly-detection/models/modelo_arritmia_final_v3.h5', 'window_size': 360, 'labels': ['A', 'L', 'N', 'R', 'V'], 'predictions': [{'index': 0, 'label': 'N', 'description': 'Batimento normal', 'confidence': 0.999931, 'probabilities': {'A': 2.3e-05, 'L': 1e-06, 'N': 0.999931, 'R': 2.9e-05, 'V': 1.6e-05}, 'source': 'signal'}]}
